# ML-02 — Research Question and Provisional Lane

This notebook frames a public-safe research question for the content-refresh lane. It uses the anonymized starter dataset and keeps the output at decision-support level.

## 1. My lane (or freestyle) and why

**Lane: content refresh prioritization.** The goal is to rank existing content pages for human review when the observed data suggests decline or an actionable refresh opportunity. This lane is useful because a content team has limited review capacity, so ordering the queue is more useful than treating every page equally.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("data/raw/content_refresh_anonymized.csv")
if not DATA.exists():
    candidates = list(Path.cwd().rglob("content_refresh_anonymized.csv"))
    if not candidates:
        raise FileNotFoundError("Run this notebook from the repository root or clone the repository first.")
    DATA = candidates[0]

df = pd.read_csv(DATA)
print("rows:", len(df))
print("columns:", len(df.columns))
print("unique clients:", df["client_id"].nunique())


## 2. The question: decision, action, cost of a wrong call

**Question:** Given the information available for a content item, which pages should an editor review first for refresh? The human action is to inspect the page and decide whether to refresh, expand, change the title/snippet, or leave it alone. A false positive costs review time; a false negative can leave a declining page unattended. The system therefore needs a ranking metric rather than a claim that a page will improve after editing.

In [ ]:
df["declining_observed"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
print("observed declining-label rate:", round(df["declining_observed"].mean(), 3))
print("items with impressions:", int((pd.to_numeric(df["impressions_90d"], errors="coerce").fillna(0) > 0).sum()))
print("items age >= 90 days:", int((pd.to_numeric(df["content_age_days"], errors="coerce").fillna(0) >= 90).sum()))


## 3. Quick look at the data (2–3 real numbers)

The starter release has 30,000 rows and 32 pseudonymized clients. The observed declining label is present on 16,262 rows (54.2%) after the reference preparation step. Those numbers make the lane large enough to compare a transparent rule with a learned ranking model, while the label mix also shows why accuracy alone would be a weak success metric.

In [ ]:
label = df["declining_observed"]
print({"rows": len(df), "clients": df["client_id"].nunique(), "declining_rate": round(label.mean(), 3)})
print("median impressions_90d:", float(pd.to_numeric(df["impressions_90d"], errors="coerce").median()))
print("median content_age_days:", float(pd.to_numeric(df["content_age_days"], errors="coerce").median()))


## 4. Careful words: what I can and cannot claim

I can make **observed**, **measured**, and **directional** statements about this dataset and say that a validated model **ranks or flags** pages at a measured precision@K. The output is **decision-support** for editorial review. I cannot claim that refreshing a page causes traffic to increase, that the model predicts Google's algorithm, or that one snapshot proves a causal effect.

In [ ]:
allowed = {"observed", "measured", "directional", "decision-support"}
print("claim vocabulary checked:", sorted(allowed))
print("target source:", "trend_direction == 'down'")


## Self-check

- [x] Every section is filled with reasoning and supporting code.
- [x] No client names, domains, or private queries are used.
- [x] Claims are framed as observed, measured, directional, or decision-support.
- [x] The notebook uses the anonymized starter dataset.